# Aya's review, 27 Sep 2026 - diagnostics

Two queries to settle items 2 and 4 with numbers rather than guesses. Both read the v73 SQL
and change nothing in production.

| Item | What CRIS showed | What Jill has | Hypothesis |
|---|---|---|---|
| 2 | 2.2 norm 394 telecasts / 218 | 393 | CRIS ran the default window (latest 12 months to 9/20/26); Jill's sheet is 9/14/25-9/13/26. One telecast in the week of 9/14-9/20/26 explains 394 vs 393. |
| 4 | 2.13 M-SU 8P-12A daypart 164 | 128 | 2.13 norms against every half hour of the daypart (premieres and repeats). Jill's daypart figure is likely repeats-only. Five ways below; whichever prints 128 is her definition. |


## Item 2 - 2.2 on Jill's window (expect NORM_TC = 393)

In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_program       = '%GREATEST MACHINES%';
SET p_season        = NULL;
SET p_stream        = 'Live+SD';
SET p_demo          = 'DEMO_AA';
SET p_include_specials = FALSE;
SET p_win_from      = '2025-09-14';
SET p_win_to        = '2026-09-13';
SET p_norm_from     = NULL;
SET p_norm_to       = NULL;
SET p_norm_start_2959 = 2000;
SET p_norm_end_2959   = 2400;
SET data_sun = (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                   WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                     AND RATING_SOURCE = $p_stream);
SET win_to   = (SELECT LEAST(COALESCE(TO_DATE($p_win_to), $data_sun), $data_sun));
SET win_from = (SELECT COALESCE(TO_DATE($p_win_from),
                  DATEADD('day', 1, DATEADD('month', -12, $win_to))));
SET norm_to   = (SELECT COALESCE(TO_DATE($p_norm_to),   $win_to));
SET norm_from = (SELECT COALESCE(TO_DATE($p_norm_from), $win_from));
SET p_premiere_only = FALSE;
SET p_exclude_curse = TRUE;
SET p_movies        = 'EXCLUDE';

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH
fy AS (
    SELECT FISCAL_YEAR, FY_START, FY_END
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
),
fact AS (
    SELECT f.*, IDENTIFIER($p_demo) AS DEMO
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT f
    WHERE ($p_network IS NULL OR f.NETWORK_CODE = $p_network)
      AND f.RATING_SOURCE = $p_stream
      AND f.BROADCAST_DATE <= $win_to
      AND ( $p_movies = 'BOTH'
         OR ($p_movies = 'EXCLUDE' AND f.MOVIE_IND = 0)
         OR ($p_movies = 'ONLY'    AND f.MOVIE_IND = 1) )
      AND ($p_include_specials OR f.CONTENT_TYPE <> 'Special')
),
target AS (
    SELECT f2.NETWORK_CODE, f2.PROGRAM_CODE, f2.CT_SEASON,
           IFF(REGEXP_LIKE(f2.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$'),
               'FY' || fy.FISCAL_YEAR, f2.CT_SEASON)                 AS NORM_KEY,
           REGEXP_LIKE(f2.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$')   AS IS_YEAR_LABEL,
           fy.FY_START, fy.FY_END
    FROM fact f2
    LEFT JOIN fy
      ON f2.BROADCAST_DATE BETWEEN fy.FY_START AND fy.FY_END
    WHERE f2.IS_FIRST_RUN_CT
      AND ($p_program IS NULL OR f2.PROGRAM_NAME ILIKE $p_program)
      AND ($p_season  IS NULL OR f2.CT_SEASON = $p_season)
      AND f2.CT_SEASON IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY f2.NETWORK_CODE, f2.PROGRAM_CODE
                               ORDER BY f2.BROADCAST_DATE DESC, f2.TELECAST_NUM DESC) = 1
),
series AS (
    SELECT f.*, t.NORM_KEY, t.IS_YEAR_LABEL
    FROM fact f
    JOIN target t
      ON  t.NETWORK_CODE = f.NETWORK_CODE
      AND t.PROGRAM_CODE = f.PROGRAM_CODE
      AND ( (NOT t.IS_YEAR_LABEL AND f.CT_SEASON = t.CT_SEASON)
         OR (t.IS_YEAR_LABEL
             AND f.BROADCAST_DATE BETWEEN t.FY_START AND t.FY_END) )
    WHERE f.IS_FIRST_RUN_CT
),
net_norm AS (
    SELECT NETWORK_CODE,
           SUM(IDENTIFIER($p_demo) * HH_MIN)
             / NULLIF(SUM(HH_MIN), 0)                    AS NET_PREM_AA,
           SUM(HH_MIN)                                   AS NET_PREM_MIN,
           COUNT(DISTINCT BROADCAST_DATE)                AS NET_NIGHTS,
           COUNT(DISTINCT PROGRAM_CODE || '|' || TELECAST_NUM
                          || '|' || BROADCAST_DATE)      AS NET_TCASTS,
           COUNT(DISTINCT PROGRAM_CODE)                  AS NET_PROGRAMMES
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE IS_AE_NETWORK
      AND ($p_network IS NULL OR NETWORK_CODE = $p_network)
      AND RATING_SOURCE = $p_stream
      AND IS_FIRST_RUN_CT
      AND BROADCAST_DATE BETWEEN $norm_from AND $norm_to
      AND ($p_norm_start_2959 IS NULL OR HH_START_2959 >= $p_norm_start_2959)
      AND ($p_norm_end_2959   IS NULL OR HH_START_2959 <  $p_norm_end_2959)
      AND ( $p_movies = 'BOTH'
         OR ($p_movies = 'EXCLUDE' AND MOVIE_IND = 0)
         OR ($p_movies = 'ONLY'    AND MOVIE_IND = 1) )
      AND ($p_include_specials OR CONTENT_TYPE <> 'Special')
      AND NOT ($p_exclude_curse AND NETWORK_CODE = 'HIST'
               AND UPPER(PROGRAM_NAME) LIKE '%CURSE%OAK ISLAND%')
    GROUP BY NETWORK_CODE
),
measured AS (
    SELECT *
    FROM series
    WHERE NOT $p_premiere_only OR IS_PREMIERE_CT = 1
),
ser AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, MAX(PROGRAM_NAME) AS PROGRAM_NAME,
           MAX(IFF($p_premiere_only, EPISODE_NAME, NULL)) AS PREMIERE_EPISODE,
           NORM_KEY,
           COUNT(*) AS TCASTS,
           MIN(BROADCAST_DATE) AS FROM_DATE,
           MAX(BROADCAST_DATE) AS TO_DATE,
           SUM(DEMO * CONTRIBUTING_DURATION) / NULLIF(SUM(CONTRIBUTING_DURATION), 0) AS SERIES_AA
    FROM measured
    GROUP BY NETWORK_CODE, PROGRAM_CODE, NORM_KEY
)
SELECT
    s.NETWORK_CODE  AS NETWORK,
    s.PROGRAM_NAME  AS PROGRAM,
    s.NORM_KEY      AS SEASON,
    IFF($p_premiere_only, 'SEASON PREMIERE', 'SEASON')             AS MEASURED,
    s.PREMIERE_EPISODE,
    s.FROM_DATE                                                    AS AIR_DATE,
    s.TO_DATE, s.TCASTS,
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    ROUND(s.SERIES_AA/1000, 0)                                     AS SERIES_AA_000,
    $norm_from AS NORM_FROM, $norm_to AS NORM_TO,
    ROUND(n.NET_PREM_AA/1000, 0)                                   AS PREMIERE_NORM_000,
    ROUND((ROUND(s.SERIES_AA/1000,0)
           / NULLIF(ROUND(n.NET_PREM_AA/1000,0),0) - 1) * 100, 1)  AS PCT_VS_NORM,
    ROUND(s.SERIES_AA/1000, 0) - ROUND(n.NET_PREM_AA/1000, 0)      AS DIFF_000,
    CASE WHEN ROUND(s.SERIES_AA/1000,0) > ROUND(n.NET_PREM_AA/1000,0) THEN 'ABOVE'
         WHEN ROUND(s.SERIES_AA/1000,0) < ROUND(n.NET_PREM_AA/1000,0) THEN 'BELOW'
         ELSE 'IN LINE' END                                        AS VERDICT,
    n.NET_TCASTS AS NORM_TCASTS, n.NET_PROGRAMMES AS NORM_PROGRAMMES
FROM ser s
JOIN net_norm n ON n.NETWORK_CODE = s.NETWORK_CODE
ORDER BY SERIES_AA_000 DESC;

## Item 4 - 2.13 daypart norm five ways (which one prints 128?)

A is what 2.13 does today. If B or D prints 128 the daypart norm needs a `p_norm_basis` parameter (ALL | REPEATS) and a decision from Jill on whether the series itself is left out.

In [ ]:
%%sql
SET p_network       = 'HIST';
SET p_program       = '%SECRET SKINWALKER RANCH%';
SET p_stream        = 'Live+SD';
SET p_demo      = 'DEMO_AA';
SET p_win_from      = NULL;
SET p_win_to        = NULL;
SET data_sun = (SELECT MAX(LATEST_FULL_SUNDAY) FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                   WHERE NETWORK_CODE = $p_network AND RATING_SOURCE = $p_stream);
SET win_to   = (SELECT LEAST(COALESCE(TO_DATE($p_win_to), $data_sun), $data_sun));
SET win_from = (SELECT COALESCE(TO_DATE($p_win_from),
                  DATEADD('day', 1, DATEADD('month', -12, $win_to))));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH daypart AS (
              SELECT 1 AS ORD, 'M-SU 8P-12A'   AS DAYPART, 2000 AS FROM_2959, 2400 AS TO_2959
    UNION ALL SELECT 2,        'M-SU 12A-6A',              2400,               2960
),
hh AS (
    SELECT h.*, d.ORD, d.DAYPART, IDENTIFIER($p_demo) AS DEMO,
           IFF(h.PROGRAM_NAME ILIKE $p_program, 1, 0) AS IS_SERIES
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT h
    JOIN daypart d
      ON h.HH_START_2959 >= d.FROM_2959 AND h.HH_START_2959 < d.TO_2959
    WHERE h.NETWORK_CODE  = $p_network
      AND h.RATING_SOURCE = $p_stream
      AND h.BROADCAST_DATE BETWEEN $win_from AND $win_to
),
ways AS (
    SELECT ORD, DAYPART, 'A. ALL PROGRAMMING (2.13 today)' AS NORM_BASIS,
           SUM(DEMO*HH_MIN)/NULLIF(SUM(HH_MIN),0) AS AA, SUM(HH_MIN) AS MIN, COUNT(*) AS HALFHRS
    FROM hh GROUP BY ORD, DAYPART
    UNION ALL
    SELECT ORD, DAYPART, 'B. REPEATS ONLY (all programmes)',
           SUM(DEMO*HH_MIN)/NULLIF(SUM(HH_MIN),0), SUM(HH_MIN), COUNT(*)
    FROM hh WHERE NOT IS_FIRST_RUN_CT GROUP BY ORD, DAYPART
    UNION ALL
    SELECT ORD, DAYPART, 'C. ALL PROGRAMMING EXCL. THE SERIES',
           SUM(DEMO*HH_MIN)/NULLIF(SUM(HH_MIN),0), SUM(HH_MIN), COUNT(*)
    FROM hh WHERE IS_SERIES = 0 GROUP BY ORD, DAYPART
    UNION ALL
    SELECT ORD, DAYPART, 'D. REPEATS ONLY EXCL. THE SERIES',
           SUM(DEMO*HH_MIN)/NULLIF(SUM(HH_MIN),0), SUM(HH_MIN), COUNT(*)
    FROM hh WHERE NOT IS_FIRST_RUN_CT AND IS_SERIES = 0 GROUP BY ORD, DAYPART
    UNION ALL
    SELECT ORD, DAYPART, 'E. THE SERIES REPEATS (what is compared)',
           SUM(DEMO*HH_MIN)/NULLIF(SUM(HH_MIN),0), SUM(HH_MIN), COUNT(*)
    FROM hh WHERE NOT IS_FIRST_RUN_CT AND IS_SERIES = 1 GROUP BY ORD, DAYPART
)
SELECT REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
       $p_network AS NETWORK, DAYPART, NORM_BASIS,
       ROUND(AA/1000, 0) AS AA_000, MIN AS MINUTES, HALFHRS,
       $win_from AS FROM_DATE, $win_to AS TO_DATE
FROM ways
ORDER BY ORD, NORM_BASIS;